<a href="https://colab.research.google.com/github/drod75/nyc_taxi_analysis/blob/main/notebooks/data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collecting & Cleaning

## Setup

In [1]:
# importing libraries, pandas for data manipulation, requests for data gathering, colab for environment variables
import pandas as pd
import geopandas
from shapely.geometry import shape
import requests
from google.colab import userdata
from google.colab import drive
from pprint import pprint as print

In [2]:
# Mount Google Drive
drive.mount('drive')

Drive already mounted at drive; to attempt to forcibly remount, call drive.mount("drive", force_remount=True).


In [3]:
# Set up the API
data_api = "https://data.cityofnewyork.us/api/v3/views/irx2-vgm4/query.json"

headers = {
    'X-App-Token': userdata.get('token'),
    "Content-Type": "application/json",
}

In [4]:
# Fetching a sample of data from the right time period
query = """
SELECT
  count(*)
WHERE
  tpep_pickup_datetime >= "2025-06-21T00:00:00"
  AND
  tpep_dropoff_datetime <= "2025-09-21T23:59:59"
LIMIT 1
"""

query_payload = {
    "query": query
}

query_response = requests.get(data_api, headers=headers, params=query_payload)
query_data = query_response.json()
print(query_data)

[{'count': '11707067'}]


In [5]:
# Collecting all the taxi data within the time period specified, if already collected use option 2
def load_raw_data(option = 1):
  if option == 1:
    query = """
    SELECT
      *
    WHERE
      tpep_pickup_datetime >= "2025-06-21T00:00:00"
      AND
      tpep_dropoff_datetime <= "2025-09-21T23:59:59"
    LIMIT 12000000
    """

    query_payload = {
        "query": query
    }

    query_response = requests.get(data_api, headers=headers, params=query_payload)
    print("All Done!!!")
    query_data = query_response.json()
    taxi_df = pd.DataFrame(query_data)
    return taxi_df
  elif option == 2:
    taxi_df = pd.read_csv("/content/drive/MyDrive/raw_taxi_summer_data.csv")
    return taxi_df
  else:
    print("Invalid option")
    return None

In [6]:
# printing length of raw initial data
taxi_df:pd.DataFrame = load_raw_data(2)
print(len(taxi_df))

/tmp/ipykernel_22993/2273556591.py:24: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  taxi_df = pd.read_csv("/content/drive/MyDrive/raw_taxi_summer_data.csv")


11707067


## Data Structure

In [7]:
# Preview the first few rows using the dataframe head function
taxi_df.head()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,:id,:version,:created_at,:updated_at,request_source
0,2,2025-06-21T00:00:00.000,2025-06-21T00:07:21.000,1.0,1.17,1.0,N,137,113,1,...,1.0,16.71,2.5,0.0,0.75,row-emei~byqv-hzqe,rv-dnr4~ztm9.kep3,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z,NaN
1,2,2025-06-21T00:00:00.000,2025-06-21T00:04:21.000,2.0,0.61,1.0,N,79,107,1,...,1.0,13.86,2.5,0.0,0.75,row-h2b5-wa6w_t924,rv-qyaa~vq94-iwvb,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z,NaN
2,2,2025-06-21T00:00:00.000,2025-06-21T00:08:00.000,NaN,1.70,NaN,NaN,74,42,0,...,1.0,2.17,NaN,NaN,0.00,row-xpsy~jicp~dcpm,rv-awpk_x4fp~ci3q,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z,NaN
3,2,2025-06-21T00:00:00.000,2025-06-21T00:29:45.000,NaN,12.49,NaN,NaN,261,244,0,...,1.0,48.18,NaN,NaN,0.75,row-9xbz-xfrr_zane,rv-efyf~7pu6_7u7f,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z,NaN
4,2,2025-06-21T00:00:00.000,2025-06-21T00:11:00.000,NaN,1.73,NaN,NaN,50,162,0,...,1.0,25.21,NaN,NaN,0.75,row-fqhg.5ceh~usg7,rv-xevg~cyge.7u66,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z,NaN


In [8]:
# Preview the last few rows using the dataframe head function
taxi_df.tail()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,:id,:version,:created_at,:updated_at,request_source
11707062,7,2025-09-21T23:58:08.000,2025-09-21T23:58:08.000,1.0,0.89,1.0,N,186,234,1,...,1.0,14.70,2.5,0.0,0.75,row-29ph~gzpu_tazy,rv-xkhe.6g4x-xzhz,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z,NaN
11707063,2,2025-09-21T23:58:15.000,2025-09-21T23:58:20.000,2.0,0.00,5.0,N,265,265,1,...,1.0,67.00,0.0,0.0,0.00,row-4g6k~vzic_7j3y,rv-zkyc-739x~wjtu,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z,NaN
11707064,2,2025-09-21T23:58:26.000,2025-09-21T23:58:30.000,1.0,0.05,5.0,N,265,265,1,...,1.0,151.00,0.0,0.0,0.00,row-jig5-i2j5_wxsw,rv-hack~bkni~b43c,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z,NaN
11707065,7,2025-09-21T23:58:54.000,2025-09-21T23:58:54.000,1.0,4.73,1.0,N,48,179,1,...,1.0,30.45,2.5,0.0,0.75,row-mt2k.x3q7~ypn4,rv-ca5i-jvv7.n7tt,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z,NaN
11707066,7,2025-09-21T23:59:43.000,2025-09-21T23:59:43.000,1.0,1.03,1.0,N,249,107,1,...,1.0,18.81,2.5,0.0,0.75,row-hgds.bskj~drqf,rv-fknd-iyz9~ecyd,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z,NaN


In [9]:
# Displaying the Dataframes shape, columns, and amount of non null values using the info function
taxi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11707067 entries, 0 to 11707066
Data columns (total 25 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   vendorid               int64  
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        float64
 4   trip_distance          float64
 5   ratecodeid             float64
 6   store_and_fwd_flag     object 
 7   pulocationid           int64  
 8   dolocationid           int64  
 9   payment_type           int64  
 10  fare_amount            float64
 11  extra                  float64
 12  mta_tax                float64
 13  tip_amount             float64
 14  tolls_amount           float64
 15  improvement_surcharge  float64
 16  total_amount           float64
 17  congestion_surcharge   float64
 18  airport_fee            float64
 19  cbd_congestion_fee     float64
 20  :id                    object 
 21  :version               object 
 22  :created_at     

In [10]:
# Drop columns not needed for the project
taxi_df.drop(columns=['request_source', 'store_and_fwd_flag',':id',':version',':created_at',':updated_at'], axis=1, inplace=True)

In [11]:
# Converting our two datetime columns to the pandas datetime type
taxi_df['tpep_pickup_datetime'] = pd.to_datetime(taxi_df['tpep_pickup_datetime'], format="mixed")
taxi_df['tpep_dropoff_datetime'] = pd.to_datetime(taxi_df['tpep_dropoff_datetime'], format="mixed")

## Missing and Duplicate Data

In [12]:
# Remove duplicate records from the dataset
taxi_df = taxi_df.drop_duplicates().reset_index(drop=True)

In [13]:
# collecting numeric columns to handle missing values
non_numeric_columns = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'store_and_fwd_flag', 'request_source']
numric_columns = [col for col in taxi_df.columns if col not in non_numeric_columns]
for col in numric_columns:
  taxi_df[col] = pd.to_numeric(taxi_df[col], errors='coerce')

In [14]:
# calculating percentage of missing data in each column
percent_missing = (taxi_df.isna().sum() / len(taxi_df)) * 100
percent_missing

,0
vendorid,0.000000
tpep_pickup_datetime,0.000000
tpep_dropoff_datetime,0.000000
passenger_count,26.594247
trip_distance,0.000000
ratecodeid,26.594247
pulocationid,0.000000
dolocationid,0.000000
payment_type,0.000000
fare_amount,0.000000


In [15]:
# Inspecting the ratecodeid for its values
print(taxi_df['ratecodeid'].value_counts(dropna=False))

ratecodeid
1.0     8059781
NaN     3113406
2.0      329229
5.0      126872
3.0       41044
4.0       34420
99.0       2306
6.0           8
Name: count, dtype: int64


In [16]:
# Fill missing values in ratecodeid
taxi_df['ratecodeid'] = taxi_df['ratecodeid'].fillna(99)

In [17]:
# Inspecting the congestion_surcharge for its values
print(taxi_df['congestion_surcharge'].value_counts(dropna=False))

congestion_surcharge
 2.50    7687339
 NaN     3113406
 0.00     727294
-2.50     179021
 1.00          5
 0.75          1
Name: count, dtype: int64


In [18]:
# Handiling congestion surchage values by replacing it with a boolean indicator
taxi_df['congestion_surcharge'].replace(
    {
      2.50: 1,
      0.00: 0,
      -2.50: 1,
      1.00: 1,
      0.75: 1
    },
    inplace=True
)

taxi_df['congestion_surcharge'] = taxi_df['congestion_surcharge'].fillna(-1)

/tmp/ipykernel_22993/305008353.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxi_df['congestion_surcharge'].replace(


In [19]:
# Inspecting the airport_fee for its values
print(taxi_df['airport_fee'].value_counts(dropna=False))

airport_fee
 0.00    7700775
 NaN     3113406
 1.75     834199
-1.75      52905
 6.75       4535
 5.00       1245
 0.50          1
Name: count, dtype: int64


In [20]:
# Handiling airport_fee values by replacing it with a boolean indicator
taxi_df['airport_fee'].replace(
    {
      0.00: 0,
      1.75: 1,
      -1.75: 1,
      6.75: 1,
      5.00: 1,
      0.50: 1
    },
    inplace=True
)

taxi_df['airport_fee'] = taxi_df['airport_fee'].fillna(-1)

/tmp/ipykernel_22993/3961128368.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxi_df['airport_fee'].replace(


In [21]:
# checking column values of when passenger count is empty
empty_passenger = taxi_df.loc[taxi_df['passenger_count'].isna()]
pmissing = (empty_passenger.isna().sum() / len(empty_passenger)) * 100
pmissing

,0
vendorid,0.0
tpep_pickup_datetime,0.0
tpep_dropoff_datetime,0.0
passenger_count,100.0
trip_distance,0.0
ratecodeid,0.0
pulocationid,0.0
dolocationid,0.0
payment_type,0.0
fare_amount,0.0


In [22]:
# checking length of data after removing duplicates and missing data
print(len(taxi_df))

11707066


## Outlier Treatment

In [23]:
# checking distribution
taxi_df[numric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
vendorid,11707066.0,1.898756,0.721782,1.00,2.00,2.00,2.00,7.00
passenger_count,8593660.0,1.314891,0.747073,0.00,1.00,1.00,1.00,9.00
trip_distance,11707066.0,6.973284,662.509687,0.00,1.07,1.92,3.90,397994.37
ratecodeid,11707066.0,27.168972,43.260031,1.00,1.00,1.00,99.00,99.00
pulocationid,11707066.0,159.865044,66.171515,1.00,114.00,161.00,231.00,265.00
dolocationid,11707066.0,159.650703,70.676549,1.00,107.00,161.00,232.00,265.00
payment_type,11707066.0,0.928752,0.796800,0.00,0.00,1.00,1.00,5.00
fare_amount,11707066.0,20.730001,96.786279,-1591.30,10.00,15.60,26.10,323800.27
extra,11707066.0,1.122846,1.813231,-7.50,0.00,0.00,2.50,38.61
mta_tax,11707066.0,0.473815,1.539577,-21.74,0.50,0.50,0.50,5243.38


In [24]:
# columns to clean
outlier_columns = ['trip_distance', 'fare_amount', 'extra', 'tip_amount', 'total_amount']

In [25]:
# create quartile barriers
Q1 = taxi_df[outlier_columns].quantile(0.25)
Q3 = taxi_df[outlier_columns].quantile(0.75)
IQR = Q3 - Q1
k = 3.0
lower_bound = Q1 - k * IQR
upper_bound = Q3 + k * IQR

In [26]:
# printing amount of outliers
within_iqr = (taxi_df[outlier_columns] >= lower_bound) & (taxi_df[outlier_columns] <= upper_bound)
print((~within_iqr).sum())

trip_distance    625183
fare_amount      208701
extra             16306
tip_amount       255635
total_amount     549943
dtype: int64


In [27]:
within_0 = (taxi_df[outlier_columns] >= 0)
print((~within_0).sum())

trip_distance         0
fare_amount      251112
extra            120912
tip_amount          414
total_amount     234635
dtype: int64


In [39]:
# iqr outlier method removal
iqr = (
    ((taxi_df[outlier_columns] >= lower_bound) & (taxi_df[outlier_columns] <= upper_bound))
).all(axis=1)

mask = (
    iqr &
    (taxi_df['trip_distance'] > 0) &
    (taxi_df['fare_amount'] > 0) &
    (taxi_df['total_amount'] > 0) &
    (taxi_df['extra'] >= 0) &
    (taxi_df['tip_amount'] >= 0) &
    (taxi_df['mta_tax'].between(0, 1.0)) &
    (taxi_df['tolls_amount'].between(0, 100.0)) &
    (taxi_df['improvement_surcharge'].between(0, 1.0))
)

taxi_df_clean = taxi_df[mask].copy()

In [40]:
taxi_df_clean.reset_index()

,index,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,0,2,2025-06-21 00:00:00,2025-06-21 00:07:21,1.0,1.17,1.0,137,113,1,9.30,1.00,0.5,1.66,0.0,1.0,16.71,1.0,0.0,0.75
1,1,2,2025-06-21 00:00:00,2025-06-21 00:04:21,2.0,0.61,1.0,79,107,1,5.80,1.00,0.5,2.31,0.0,1.0,13.86,1.0,0.0,0.75
2,4,2,2025-06-21 00:00:00,2025-06-21 00:11:00,NaN,1.73,99.0,50,162,0,16.26,0.00,0.5,4.20,0.0,1.0,25.21,-1.0,-1.0,0.75
3,5,2,2025-06-21 00:00:00,2025-06-21 00:13:00,NaN,1.07,99.0,79,79,0,8.99,0.00,0.5,0.00,0.0,1.0,13.74,-1.0,-1.0,0.75
4,6,1,2025-06-21 00:00:01,2025-06-21 00:02:33,1.0,0.60,1.0,263,262,1,5.10,3.50,0.5,3.05,0.0,1.0,13.15,1.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10403567,11707058,1,2025-09-21 23:57:28,2025-09-21 23:59:41,1.0,0.90,1.0,50,246,1,5.80,4.25,0.5,2.00,0.0,1.0,13.55,1.0,0.0,0.75
10403568,11707060,7,2025-09-21 23:58:02,2025-09-21 23:58:02,2.0,2.38,1.0,142,68,1,14.20,0.00,0.5,3.99,0.0,1.0,23.94,1.0,0.0,0.75
10403569,11707061,7,2025-09-21 23:58:08,2025-09-21 23:58:08,1.0,0.89,1.0,186,234,1,6.50,0.00,0.5,2.45,0.0,1.0,14.70,1.0,0.0,0.75
10403570,11707064,7,2025-09-21 23:58:54,2025-09-21 23:58:54,1.0,4.73,1.0,48,179,1,24.70,0.00,0.5,0.00,0.0,1.0,30.45,1.0,0.0,0.75


In [41]:
taxi_df_clean.head()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,2,2025-06-21 00:00:00,2025-06-21 00:07:21,1.0,1.17,1.0,137,113,1,9.30,1.0,0.5,1.66,0.0,1.0,16.71,1.0,0.0,0.75
1,2,2025-06-21 00:00:00,2025-06-21 00:04:21,2.0,0.61,1.0,79,107,1,5.80,1.0,0.5,2.31,0.0,1.0,13.86,1.0,0.0,0.75
4,2,2025-06-21 00:00:00,2025-06-21 00:11:00,NaN,1.73,99.0,50,162,0,16.26,0.0,0.5,4.20,0.0,1.0,25.21,-1.0,-1.0,0.75
5,2,2025-06-21 00:00:00,2025-06-21 00:13:00,NaN,1.07,99.0,79,79,0,8.99,0.0,0.5,0.00,0.0,1.0,13.74,-1.0,-1.0,0.75
6,1,2025-06-21 00:00:01,2025-06-21 00:02:33,1.0,0.60,1.0,263,262,1,5.10,3.5,0.5,3.05,0.0,1.0,13.15,1.0,0.0,0.00


In [42]:
taxi_df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10403572 entries, 0 to 11707065
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   vendorid               int64         
 1   tpep_pickup_datetime   datetime64[ns]
 2   tpep_dropoff_datetime  datetime64[ns]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   ratecodeid             float64       
 6   pulocationid           int64         
 7   dolocationid           int64         
 8   payment_type           int64         
 9   fare_amount            float64       
 10  extra                  float64       
 11  mta_tax                float64       
 12  tip_amount             float64       
 13  tolls_amount           float64       
 14  improvement_surcharge  float64       
 15  total_amount           float64       
 16  congestion_surcharge   float64       
 17  airport_fee            float64       
 18  cbd_congestion_fee     fl

In [43]:
taxi_df_clean.describe().T

,count,mean,min,25%,50%,75%,max,std
vendorid,10403572.0,1.894971,1.0,2.0,2.0,2.0,7.0,0.740538
tpep_pickup_datetime,10403572,2025-08-07 01:08:12.065357568,2025-06-21 00:00:00,2025-07-14 11:42:58,2025-08-06 09:35:36,2025-09-01 13:20:56,2025-09-21 23:59:43,NaN
tpep_dropoff_datetime,10403572,2025-08-07 01:23:16.926605312,2025-06-21 00:00:29,2025-07-14 11:57:31.249999872,2025-08-06 09:52:18,2025-09-01 13:33:57.500000,2025-09-21 23:59:58,NaN
passenger_count,7642617.0,1.295263,0.0,1.0,1.0,1.0,8.0,0.729497
trip_distance,10403572.0,2.737719,0.01,1.1,1.87,3.4,12.39,2.453057
ratecodeid,10403572.0,27.033745,1.0,1.0,1.0,99.0,99.0,43.268431
pulocationid,10403572.0,161.492621,1.0,113.0,161.0,233.0,265.0,67.057217
dolocationid,10403572.0,160.589919,1.0,107.0,162.0,233.0,265.0,70.194338
payment_type,10403572.0,0.878595,0.0,0.0,1.0,1.0,4.0,0.695533
fare_amount,10403572.0,18.051241,0.01,10.0,14.9,23.3,74.4,11.061724


In [44]:
print(len(taxi_df_clean))

10403572


## Taxi Zones Data

In [45]:
# Querying for taxi zones data
zones_query = {
    "query":"""
    SELECT
      *
    LIMIT 500
    """
}
zones_response = requests.get(
    "https://data.cityofnewyork.us/api/v3/views/8meu-9t5y/query.json",
    headers=headers,
    params=zones_query
  )
zone_data = zones_response.json()
print("All done!")

'All done!'


In [46]:
zone_df = pd.DataFrame(zone_data)
zone_df["the_geom"] = zone_df["the_geom"].apply(lambda g: shape(g) if isinstance(g, dict) else None)
zone_df = geopandas.GeoDataFrame(zone_df, geometry="the_geom", crs="EPSG:4326")

In [47]:
# Remove metadata columns
zone_df.drop(columns=[':id', ':version', ':created_at', ':updated_at'], axis=1, inplace=True)

In [48]:
# fixing datatypes of three specific columns specified below
zone_df['shape_leng'] = zone_df['shape_leng'].astype("float")
zone_df['shape_area'] = zone_df['shape_area'].astype("float")
zone_df['locationid'] = zone_df['locationid'].astype("int")

In [49]:
# checking our dataframes structure again
zone_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   the_geom    263 non-null    geometry
 1   shape_leng  263 non-null    float64 
 2   shape_area  263 non-null    float64 
 3   zone        263 non-null    object  
 4   locationid  263 non-null    int64   
 5   borough     263 non-null    object  
dtypes: float64(2), geometry(1), int64(1), object(2)
memory usage: 12.5+ KB


## Borough Validation

In [50]:
# importing datasets from Google Drive
zone_df = geopandas.read_file("/content/drive/MyDrive/taxi_zones_data.json")

In [51]:
# merging out tables
taxi_zone_df = pd.merge(taxi_df_clean, zone_df, left_on='pulocationid', right_on='locationid', how='left')
taxi_zone_df = pd.merge(taxi_zone_df, zone_df, left_on='dolocationid', right_on='locationid', how='left')
taxi_zone_df.rename(columns={'borough_x': 'pickup_borough', 'borough_y': 'dropoff_borough'}, inplace=True)
taxi_zone_df.drop(columns=['locationid_x', 'locationid_y'], inplace=True)
taxi_zone_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10413976 entries, 0 to 10413975
Data columns (total 29 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   vendorid               int64         
 1   tpep_pickup_datetime   datetime64[ns]
 2   tpep_dropoff_datetime  datetime64[ns]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   ratecodeid             float64       
 6   pulocationid           int64         
 7   dolocationid           int64         
 8   payment_type           int64         
 9   fare_amount            float64       
 10  extra                  float64       
 11  mta_tax                float64       
 12  tip_amount             float64       
 13  tolls_amount           float64       
 14  improvement_surcharge  float64       
 15  total_amount           float64       
 16  congestion_surcharge   float64       
 17  airport_fee            float64       
 18  cbd_congestion_fee  

In [52]:
# checking pickup borough values
taxi_zone_df['pickup_borough'].value_counts(dropna=False)

,count
pickup_borough,
Manhattan,9292351
Queens,662230
Brooklyn,381760
Bronx,60363
NaN,16432
Staten Island,709
EWR,131


In [53]:
# checking dropoff borough values
taxi_zone_df['dropoff_borough'].value_counts(dropna=False)

,count
dropoff_borough,
Manhattan,9176061
Brooklyn,566551
Queens,539446
Bronx,96345
NaN,34850
Staten Island,532
EWR,191


In [54]:
# checking pickup borough values when NaN
taxi_zone_df.loc[taxi_zone_df['pickup_borough'].isna(), ['pulocationid', 'pickup_borough', 'dolocationid','dropoff_borough']]

,pulocationid,pickup_borough,dolocationid,dropoff_borough
178,264,NaN,162,Manhattan
1309,264,NaN,264,NaN
1344,264,NaN,264,NaN
1673,264,NaN,219,Queens
1832,264,NaN,145,Queens
...,...,...,...,...
10410415,264,NaN,186,Manhattan
10411039,264,NaN,264,NaN
10411157,264,NaN,53,Queens
10412418,264,NaN,264,NaN


In [55]:
# value counts of location id where borough is null
taxi_zone_df.loc[taxi_zone_df['pickup_borough'].isna(), 'pulocationid'].value_counts()


,count
pulocationid,
264,14785
265,1286
57,361


In [56]:
# checking pickup borough values when NaN
taxi_zone_df.loc[taxi_zone_df['dropoff_borough'].isna(), ['pulocationid', 'pickup_borough', 'dolocationid','dropoff_borough']]

,pulocationid,pickup_borough,dolocationid,dropoff_borough
92,100,Manhattan,57,NaN
247,234,Manhattan,265,NaN
381,249,Manhattan,264,NaN
831,263,Manhattan,264,NaN
1151,148,Manhattan,264,NaN
...,...,...,...,...
10412094,170,Manhattan,264,NaN
10412130,132,Queens,265,NaN
10412396,132,Queens,265,NaN
10412418,264,NaN,264,NaN


In [57]:
# value counts of location id where borough is null
taxi_zone_df.loc[taxi_zone_df['dropoff_borough'].isna(), 'dolocationid'].value_counts()

,count
dolocationid,
265,17335
264,17014
57,500
105,1


In [58]:
# value counts of location id where borough is ewr
taxi_zone_df.loc[taxi_zone_df['pickup_borough'] == 'EWR', 'pulocationid'].value_counts()


,count
pulocationid,
1,131


In [59]:
# value counts of location id where borough is ewr
taxi_zone_df.loc[taxi_zone_df['dropoff_borough'] == 'EWR', 'dolocationid'].value_counts()

,count
dolocationid,
1,191


In [60]:
# dropping taxi data where location id is specific values 265, 264, 57, 105, 1
taxi_df_clean = taxi_df_clean.loc[~taxi_df_clean['pulocationid'].isin([265, 264, 57, 105, 1])]
taxi_df_clean = taxi_df_clean.loc[~taxi_df_clean['dolocationid'].isin([265, 264, 57, 105, 1])]

In [61]:
# double checking ids
taxi_df_clean.loc[taxi_df_clean['pulocationid'].isin([265, 264, 57, 105, 1])]

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee


In [62]:
# double checking ids
taxi_df_clean.loc[taxi_df_clean['dolocationid'].isin([265, 264, 57, 105, 1])]

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee


In [64]:
taxi_df_clean.reset_index()

,index,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,0,2,2025-06-21 00:00:00,2025-06-21 00:07:21,1.0,1.17,1.0,137,113,1,9.30,1.00,0.5,1.66,0.0,1.0,16.71,1.0,0.0,0.75
1,1,2,2025-06-21 00:00:00,2025-06-21 00:04:21,2.0,0.61,1.0,79,107,1,5.80,1.00,0.5,2.31,0.0,1.0,13.86,1.0,0.0,0.75
2,4,2,2025-06-21 00:00:00,2025-06-21 00:11:00,NaN,1.73,99.0,50,162,0,16.26,0.00,0.5,4.20,0.0,1.0,25.21,-1.0,-1.0,0.75
3,5,2,2025-06-21 00:00:00,2025-06-21 00:13:00,NaN,1.07,99.0,79,79,0,8.99,0.00,0.5,0.00,0.0,1.0,13.74,-1.0,-1.0,0.75
4,6,1,2025-06-21 00:00:01,2025-06-21 00:02:33,1.0,0.60,1.0,263,262,1,5.10,3.50,0.5,3.05,0.0,1.0,13.15,1.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10359629,11707058,1,2025-09-21 23:57:28,2025-09-21 23:59:41,1.0,0.90,1.0,50,246,1,5.80,4.25,0.5,2.00,0.0,1.0,13.55,1.0,0.0,0.75
10359630,11707060,7,2025-09-21 23:58:02,2025-09-21 23:58:02,2.0,2.38,1.0,142,68,1,14.20,0.00,0.5,3.99,0.0,1.0,23.94,1.0,0.0,0.75
10359631,11707061,7,2025-09-21 23:58:08,2025-09-21 23:58:08,1.0,0.89,1.0,186,234,1,6.50,0.00,0.5,2.45,0.0,1.0,14.70,1.0,0.0,0.75
10359632,11707064,7,2025-09-21 23:58:54,2025-09-21 23:58:54,1.0,4.73,1.0,48,179,1,24.70,0.00,0.5,0.00,0.0,1.0,30.45,1.0,0.0,0.75


In [65]:
print(len(taxi_df_clean))

10359634


## Export

In [66]:
# Export the processed taxi records and spatial zone lookup tables to Google Drive
taxi_df_clean.to_csv("/content/drive/MyDrive/taxi_summer_data.csv", index=False)
zone_df.to_file("/content/drive/MyDrive/taxi_zones_data.json", index=False)